In [2]:
from scipy.linalg import svd
import numpy as np

def reshape_matrix(matrix, L):
    """Reshapes the 2^L x 2^L matrix into a tensor of shape (2, 2, ..., 2, 2)."""
    dim = 2**L
    return matrix.reshape([2] * (2 * L))

def matrix_to_mpo(matrix, L):
    """Converts a 2^L x 2^L matrix into an MPO representation."""
    tensor = reshape_matrix(matrix, L)
    mpo = []
    current_tensor = tensor

    for i in range(L - 1):
        d1 = current_tensor.shape[0]
        d2 = current_tensor.shape[-1]
        d_mid = np.prod(current_tensor.shape[1:-1])
        current_tensor = current_tensor.reshape(d1 * d_mid, d2)
        
        U, S, Vh = svd(current_tensor, full_matrices=False)
        
        U = U.reshape((d1, -1, d_mid))
        
        if i == 0:
            mpo.append(U)
        else:
            mpo.append(U.reshape((mpo[-1].shape[-1], d1, -1)))
        
        current_tensor = np.dot(np.diag(S), Vh)
        current_tensor = current_tensor.reshape((len(S), d2))
        
        if i < L - 2:
            current_tensor = current_tensor.reshape((len(S), 2, -1))
    
    mpo.append(current_tensor.reshape((mpo[-1].shape[-1], 2, 1, 2)))
    return mpo

def state_to_mps(state, L):
    """Converts a 2^L state vector into an MPS representation."""
    mps = []
    current_tensor = state.reshape([2] * L)  # Initial shape: (2, 2, ..., 2) with L dimensions
    
    for i in range(L - 1):
        d1 = current_tensor.shape[0]  # First dimension size
        d_mid = np.prod(current_tensor.shape[1:])  # Product of intermediate dimensions
        
        # Reshape current_tensor to a 2D matrix of shape (d1, d_mid)
        current_tensor = current_tensor.reshape(d1, d_mid)
        
        # Perform SVD on the reshaped tensor
        U, S, Vh = svd(current_tensor, full_matrices=False)
        # Shapes after SVD:
        # U: (d1, k) where k = min(d1, d_mid)
        # S: (k,)
        # Vh: (k, d_mid)
        
        # Reshape U to form the MPS tensor for this site
        U = U.reshape((d1, -1, len(S)))  # Shape: (d1, 1, len(S)) initially
        
        if i == 0:
            mps.append(U)
        else:
            mps.append(U.reshape((mps[-1].shape[-1], d1, -1)))  # Shape: (prev_k, d1, len(S))
        
        # Update current_tensor for the next iteration
        current_tensor = np.dot(np.diag(S), Vh)
        # Shape: (len(S), d_mid)
        
        if i < L - 2:
            current_tensor = current_tensor.reshape((len(S), 2, -1))
            # Shape: (len(S), 2, rest of the dimensions)
    
    # Add the last tensor
    mps.append(current_tensor.reshape((mps[-1].shape[-1], 2, 1)))
    # Shape: (prev_k, 2, 1)
    
    return mps


def apply_mpo_to_mps(mpo, mps, L):
    """Applies an MPO to an MPS, resulting in a new MPS."""
    new_mps = [None] * L
    
    for i in range(L):
        mpo_tensor = mpo[i]
        mps_tensor = mps[i]
        
        if i == 0:
            new_tensor = np.tensordot(mpo_tensor, mps_tensor, axes=([1], [1]))
            new_tensor = new_tensor.transpose(1, 0, 2, 3).reshape(new_tensor.shape[1], -1, new_tensor.shape[3])
        elif i == L - 1:
            new_tensor = np.tensordot(mpo_tensor, mps_tensor, axes=([2], [2]))
            new_tensor = new_tensor.transpose(0, 2, 1, 3).reshape(new_tensor.shape[0], new_tensor.shape[1], -1)
        else:
            new_tensor = np.tensordot(mpo_tensor, mps_tensor, axes=([2], [1]))
            new_tensor = new_tensor.transpose(0, 3, 1, 2, 4).reshape(new_tensor.shape[0], new_tensor.shape[1], -1, new_tensor.shape[4])
        
        new_mps[i] = new_tensor
    
    return new_mps

# Parameters
L = 3  # Length of the spin chain

# Create a random unitary matrix
unitary_matrix = np.random.rand(2**L, 2**L) + 1j * np.random.rand(2**L, 2**L)
unitary_matrix, _ = np.linalg.qr(unitary_matrix)  # Ensure the matrix is unitary

# Convert the unitary matrix into an MPO
unitary_mpo = matrix_to_mpo(unitary_matrix, L)

# Create a random initial state
initial_state = np.random.rand(2**L) + 1j * np.random.rand(2**L)
initial_state /= np.linalg.norm(initial_state)  # Normalize the state

# Convert the initial state into an MPS
initial_mps = state_to_mps(initial_state, L)

# Apply the unitary MPO to the initial MPS
final_mps = apply_mpo_to_mps(unitary_mpo, initial_mps, L)

# Print the resulting MPS tensors
for i, tensor in enumerate(final_mps):
    print(f"MPS tensor at site {i}:")
    print(tensor.shape)

ValueError: einstein sum subscripts string contains too many subscripts for operand 1